# Lexos Clustermap Tutorial

Unlike a simple dendrogram that only clusters documents, a Clustermap allows you to simultaneously cluster both your documents and the terms within them. This provides a rich, two-dimensional view of where specific terms are concentrated across your documents and how those concentrations lead to natural groupings.

We'll begin by importing some data and creating a DTM.

In [ ]:
from lexos.dtm import DTM
from lexos.io.loader import Loader
from lexos.tokenizer import Tokenizer

# Load some text files and set their names
files = [
    "FilesToUse/AI_The_Hermit.txt",
    "FilesToUse/AI_The_Invention.txt",
    "FilesToUse/HenryWP_ThePirate_Short.txt",
]

loader = Loader()
loader.load(files)
loader.names = ["AI1", "AI2", "Henry"]

# Tokenize the loaded documents
tokenizer = Tokenizer()
docs = list(tokenizer.make_docs(texts=loader.texts))
labels = loader.names

print(f"Loaded {len(docs)} documents with labels: {labels}")

# Create a Document-Term Matrix (DTM)
dtm = DTM()
dtm(docs=docs, labels=labels)

print(f"DTM created with {dtm.to_df().shape[1]} documents and {dtm.to_df().shape[0]} unique terms.")


### Generating the Clustermap 

A clustermap is produced using Seaborn, a Python data visualization library built on top of `matplotlib`.

When we create the clustermap, we need to tell it how to measure distances and how to arrange the clusters. Here are the key parameters you can adjust:
 
- `dtm`: This is our "linguistic spreadsheet" (`dtm`) that we created in the previous step. It's the essential input for the tree.
- `metric`: This tells the dendrogram how to measure the "distance" or dissimilarity between your documents. Shorter distances mean more similar documents.
    - `"euclidean"` (default): Think of this as the "straight-line" distance between two points on a graph. It's good for general comparisons but can be sensitive to the overall length of documents (longer documents might naturally have larger term counts, increasing their "distance").
    - `"cosine"`: Imagine each document as an arrow pointing in a specific linguistic "direction." Cosine similarity measures how much these arrows point in the same direction. If they point almost identically, the documents are very similar, even if one document is much longer than another. This is often an excellent choice for text analysis as it focuses on stylistic or thematic *direction* rather than raw word counts.
    - `"cityblock"` (also called Manhattan distance): Imagine moving on a city grid where you can only go along streets (no diagonal shortcuts). This distance is the sum of the absolute differences for each term between two documents. Useful when the individual differences in term counts are important.
    * Many other metrics are available (e.g., "jaccard", "chebyshev"). You can find a full list in the SciPy documentation for `scipy.spatial.distance.pdist`.

- `method`: Once we've measured distances, this method determines how individual documents (or existing clusters of documents) are joined together to form larger branches and clusters in the tree.
    - `"average"` (default): When combining two clusters, this method considers the average distance between *all* pairs of documents in the two clusters. It tends to produce well-balanced clusters.
    - `"single"`: Joins clusters based on the *closest* pair of documents between them. This can sometimes lead to "chaining," where documents connect one after another, forming long, straggly branches.
    - `"complete"`: Joins clusters based on the *farthest* pair of documents between them. This tends to produce more compact, spherical clusters, ensuring all documents within a cluster are relatively similar to each other.
    - `"ward"`: This method aims to minimize the increase in "variance" (or spread) within clusters when they are merged. It tries to make clusters that are as "tight" and internally similar as possible. Often produces intuitive and well-structured clusters.
    * Many other methods are available. You can find a full list in the SciPy documentation for `scipy.cluster.hierarchy.linkage`.
- `labels`: This is simply the list of descriptive names for your documents (e.g., "Poe", "Lippard") that we defined earlier. These will appear as the leaves (endpoints) on your tree.
- `z_score`: Crucial for normalizing your data on the heatmap. It standardizes the values within each row (documents) or column (terms). If the value is set to `None`, the heatmap shows raw frequencies (or whatever your DTM contains). The setting `0` standardizes each row (document) by subtracting its mean and dividing by its standard deviation. This highlights how *terms vary within a single document* relative to that document's average term frequency. Useful for comparing patterns across documents regardless of their length. The setting `1` standardizes each column (term) by subtracting its mean and dividing by its standard deviation. This highlights how *a single term's frequency varies across different documents* relative to that term's average frequency. Useful for seeing which documents use a term more or less than average.
- `standard_scale`: Similar to `z_score`, but scales to a specific range (usually 0 to 1). The setting `0` scales each row (document) so its minimum value is 0 and its maximum is 1. The setting `1` scales each column (term) so its minimum is 0 and its maximum is 1.
- `cmap`: This sets the color scheme (colormap) for the heatmap. It determines which colors represent low values and which represent high values. The default setting "vlag" is a diverging colormap (red/blue), which is good for showing values around a center point (especially after `z_score` scaling). Other good general-purpose colormaps are "viridis" and "coolwarm". You can find listings of other [`matplotlib`](https://matplotlib.org/stable/gallery/color/colormap_reference.html) and [`seaborn`](https://seaborn.pydata.org/tutorial/color_palettes.html) colormaps online.
- `hide_upper`: Setting the value to `True` removes the dendrogram above the heatmap. Useful if you are not interested in the clustering of columns/terms.
- `hide_side`: Setting the value to `True` removes the dendrogram to the left of the heatmap. Useful if you are not interested in the clustering of rows/documents.
- `row_cluster`: Perform clustering on rows (documents). Default is `True`. Along with `col_cluster`, this setting is useful if you have a specific ordering in mind for comparison, or if you've pre-computed a linkage.
- `col_cluster`: Perform clustering on columns (terms). Default is `False`. If `False`, items will be displayed in their original order.
- `row_colors`: Allows you to add colored strips alongside the rows, which can be used to visually group or categorize your documents. Provide a list of colors (e.g., `['red', 'blue', 'green']`). The list should match the number of documents/terms. You can also use a named `seaborn` palette (e.g., `"husl"`). Setting the value to "default" will use `seaborn.husl_palette(8, s=0.45)`. This setting is great for adding metadata! For example, if you have two categories of documents (e.g., "male authors" vs. "female authors"), you could assign a color to each category to see if your clustering aligns with these external factors.
- `col_colors`:Allows you to add colored strips alongside the columns, which can be used to visually group or categorize your terms. Setting values are the same as for `row_colors`.
- `showfig`: Controls whether the generated tree figure is displayed directly in this Jupyter Notebook cell.
    - `True`: The tree will appear right below the code cell.
    - `False` (default): The tree will not be shown immediately. This is useful if you just want to save the figure to a file without displaying it in the notebook. If you set `show=False`, remember to call `dendrogram.showfig()` later to display it.
- `title`: Adds a title to your dendrogram plot.
- `figsize`: A tuple `(width, height)` in inches to set the size of the overall figure. For example, `(12, 8)` for a wider and taller plot.

Let's generate our first clustermap! We'll start with common parameters, but feel free to come back and experiment with them.


In [ ]:
# Import the ClusterMap class
from lexos.cluster.clustermap import ClusterMap

# Create an instance of the ClusterMap object
cm = ClusterMap()

# Generate the ClusterMap.
fig = cm(
    dtm=dtm,
    labels=labels,
    metric="euclidean",
    method="average",
    z_score=1,          # standardize each column (term)
    cmap="viridis",
    figsize=(18, 10),   # Increase figure width for many columns
    title="ClusterMap of Literary Texts by Term Frequency",
    hide_upper=False,
    hide_side=False,
    showfig=True,
    # Uncomment and try these for extra insights:
    # col_cluster=False, # Set to False if you want terms in their original order
    # dendrogram_ratio=(0.15, 0.25) # Adjust space for dendrograms
)

### Interpreting Your ClusterMap: Heatmap, Dendrograms, and What They Mean

The Clustermap provides a wealth of information at a glance. Let's break down how to interpret it:

- The Heatmap is the core of the Clustermap. Each cell represents the frequency (or scaled frequency, if you used `z_score` or `standard_scale`) of a particular term in a particular document. The color of each cell indicates the value. Remember that each row is a document and each column is a term.
- The two dendrograms show the clustering. The left dendrogram shows how your documents are grouped based on their shared term usage patterns. Documents that are closer together on this tree are more linguistically similar. The top dendrogram shows how your terms are grouped based on their co-occurrence patterns across documents. Terms that are closer together here tend to appear similarly across your corpus.
- Shorter horizontal branches mean greater similarity/closer relationship.

You can interpret the diagram using the following procedure.

1.  Examine the dendrogram on the left to identify which documents cluster together.
2.  For a given cluster of documents, look at the corresponding rows in the heatmap. What terms (columns) are particularly "hot" (red) or "cold" (blue) within that cluster? This reveals the vocabulary that defines that group of texts.
3.  Examine the dendrogram on the top to identify which terms cluster together.
4.  For a given cluster of terms, look at the corresponding columns in the heatmap. Which documents are particularly "hot" or "cold" for these terms? 
5.  Look for Patterns. Do authors of a certain period cluster together? Do texts from a particular genre show similar term usage patterns? Do specific themes correspond to clusters of terms?

The Clustermap allows you to go beyond simple similarity and pinpoint *which specific terms* are driving those similarities and differences.

### Customizing the Clustermap

The Lexos `ClusterMap` class uses `seaborn` and `matplotlib` to create the plot, giving you extensive control over its appearance. You can adjust colors, sizes, titles, and even hide parts of the plot to emphasize your findings.

#### Adjusting Figure Size

The `figsize` parameter in the `ClusterMap` call is critical for making your plot readable, especially with many documents or terms.

```python
# Example of adjusting figsize
# fig = cm(
#     dtm=dtm_df,
#     labels=labels,
#     # ... other parameters ...
#     figsize=(15, 12), # Make it wider and taller
#     showfig=True
# )

### Changing the Color Map (cmap)
Experiment with different colormaps to find one that best highlights your data's patterns.

- Diverging (good for z_score data): "vlag", "coolwarm", "RdBu" (red-blue)
- Sequential (good for raw counts, high to low): "viridis", "plasma", "YlGnBu" (yellow-green-blue)

Uncomment and run the cell below to see how this works (note that other parameters are the defaults).

In [ ]:
# Example of changing cmap
# fig = cm(
#     dtm=dtm,
#     labels=labels,
#     cmap="viridis", # Change to a sequential colormap
#     showfig=True
# )

### Adding Row/Column Colors

In [ ]:
# !!! This cell does not work

# Example: If 'Poe_Usher' and 'Lippard_Bel' are 'Gothic' and others are 'Adventure'
# Make sure the list of colors matches your 'labels' list in order
document_categories = ["Gothic", "Gothic", "Adventure", "Adventure", "Adventure", "Adventure"]
unique_categories = list(set(document_categories))
color_map = {"Gothic": "purple", "Adventure": "orange"}
assigned_colors = [color_map[cat] for cat in document_categories]

# fig = cm(
#     dtm=dtm,
#     labels=labels,
#     # ... other parameters ...
#     row_colors=assigned_colors, # Use your custom list of colors
#     # Or use a seaborn palette:
#     # row_colors="deep",
#     showfig=True
# )


### Hiding Dendrograms

If you prefer a simpler visual, you can hide the dendrograms.

In [ ]:
# Example: Hiding the upper (column) dendrogram
# cm = ClusterMap()
# fig = cm(
#     dtm=dtm,
#     labels=labels,
#     metric="euclidean",
#     method="average",
#     z_score=1,          # standardize each column (term)
#     cmap="viridis",
#     figsize=(18, 10),   # Increase figure width for many columns
#     title="ClusterMap of Literary Texts by Term Frequency",
#     hide_upper=True,
#     hide_side=False,
#     showfig=True,
# )

### Saving Your Clustermap
After you've generated your clustermap, you'll likely want to save it as an image for reports or presentations. The save function lets you do this easily. Just provide a file path, and it'll save the image. You can specify different file formats by changing the extension (e.g., .png, .jpg, .pdf, .svg). 
- `dpi` is a helpful argument that allows you to change the resolution of the image. A dpi of 300 is considered high resolution.

In [ ]:
# Save the clustermap with a higher resolution
# cm.save("my_clustermap_high_res.png", dpi=300)

## Plotly Clustermaps

Plotly clustermaps are somewhat experimental and may not render plots that are as informative as Seaborn clustermaps. One advantage they have is that, instead of providing labels for each document at the bottom of the graph, they provide the document labels on the `x` and `y` axes, as well as the `z` (distance) score in the hovertext. This allows you to mouse over individual sections of the heatmap to see which documents are represented by that particular section.

We'll start by getting some fresh data:

In [ ]:
from lexos.dtm import DTM
from lexos.io.loader import Loader
from lexos.tokenizer import Tokenizer

# Load some text files and set their names
files = [
    "FilesToUse/Poe_FallOfHouseUsher_1839.txt",
    "FilesToUse/Lippard_BelOfPrairieEden.txt",
    "FilesToUse/Irving_RipVanWInkle.txt",
    "FilesToUse/HenryWP_ThePirate.txt",
]
loader = Loader()
loader.load(files)
loader.names = ["Poe", "Lippard", "Irving", "Henry"]

# Tokenize the loaded documents
tokenizer = Tokenizer()
docs = list(tokenizer.make_docs(texts=loader.texts))
labels = loader.names

print(f"Loaded {len(docs)} documents with labels: {labels}")

# Create a Document-Term Matrix (DTM)
dtm = DTM()
dtm(docs=docs, labels=labels)

print(f"DTM created with {dtm.to_df().shape[1]} documents and {dtm.to_df().shape[0]} unique terms.")


Now we'll generate the Plotly clustermap.

Note that once the clustermap plot has been generated, it is inadvisable to use the modebar zoom and pan buttons because this tends to separate the heatmap from the dendrogram leaves. In the future, these buttons may be removed.

In [ ]:
from lexos.cluster.plotly_clustermap import PlotlyClustermap

# Create an instance of the PlotlyClustermap object
clustermap = PlotlyClustermap()

# Generate the Plotly Clustermap.
clustermap(
    dtm=dtm,
    labels=labels,
    metric="euclidean",  # Try "cosine" for stylistic comparisons, or "cityblock"
    method="average",    # Try "ward" for compact clusters, or "complete"
    colorscale="Viridis",# Choose a color scheme for the heatmap (e.g., "Blues", "RdBu")
    width=600,           # Set width of the plot in pixels
    height=450,          # Set height of the plot in pixels
    title="Document Similarity Clustermap",
    showfig=True         # Display the figure in the notebook
)

# If showfig=False was used, you can explicitly show the figure like this:
# clustermap.show()

Customizing Your Visualization
 
 The Lexos `PlotlyClustermap` class generates an interactive Plotly figure, giving you a lot of flexibility for customization beyond the initial parameters. You can adjust dimensions, hide dendrograms, and even control the colorscale.
 
 #### Adjusting Dimensions and Hiding Dendrograms
 
 You can change the `width` and `height` of the plot to make it larger or smaller. You can also hide the top or side dendrograms if you only want to focus on the heatmap or a single clustering view.


In [ ]:
clustermap_custom = PlotlyClustermap()
clustermap_custom(
    dtm=dtm,
    labels=labels,
    metric="cosine", # Using cosine for a different perspective
    method="ward",   # Using ward for compact clusters
    width=700,
    height=600,
    hide_upper=False, # Set to True to hide the top dendrogram
    hide_side=False,  # Set to True to hide the side dendrogram
    colorscale="Blues", # Try a different colorscale
    title="Clustermap: Cosine Distance, Ward Linkage (Blue Scale)",
    showfig=True
)


### Saving Your Clustermap
 
 Once you're happy with your Plotly Dendrogram, you'll likely want to save it as an interactive HTML file or a static image for reports, presentations, or simply for your records.

You have several options. In the Plotly toolbar, there is a "Download plot as png" option to save the plot as a static `.png` file. You can also save the the image to a static file programmatically by calling `PlotlyClustermap.write_image()`. Just provide a file name (including the extension), and it will save the image. You can choose different file formats by changing the extension (e.g., `.png`, `.jpg`, `.pdf`, `.svg`). This is a wrapper around Plotly's [`write_image()`](https://plotly.github.io/plotly.py-docs/generated/plotly.io.write_image.html) function and accepts all the same arguments.

Plotly figures are highly interactive when saved as HTML, allowing you to zoom, pan, and hover over data points in your saved file. If you wish to save your diagram as an HTML file, call `PlotlyClustermap.write_html()`. This is a wrapper around Plotly's [`write_html()`](https://plotly.github.io/plotly.py-docs/generated/plotly.io.write_html.html) function and accepts all the same arguments.

Note that `write_image()` and `write_html()` have parallel `to_image()` and `to_html()` methods that allow you to assign the results to a variable, rather than saving to a file. 

An example is given below:

In [ ]:
# Save as an interactive HTML file
# clustermap_custom.write_html("my_clustermap_analysis.html")

### Troubleshooting

**"My clustermap looks blank or I get errors about the DTM."**

- Ensure your `dtm` was created successfully in Section 4 and contains terms. If your documents are very short or very similar, the DTM might be sparse or have issues that prevent clustering.
- Check `showfig=True`: Make sure you set `showfig=True` when calling the `PlotlyClustermap` instance, or explicitly call `clustermap.show()` after creating it.
 
**"My documents don't cluster the way I expected!"**

- Experiment with the `metric` and `method` settings. Different distance metrics and linkage methods will highlight different types of similarity. `"cosine"` is often excellent for stylistic comparisons in text. `"ward"` or `"average"` are common and effective linkage methods.
- Adjust your DTM pre-processing. The content of your DTM directly influences the clustering. Try different combinations of `stop_words`, `lemmatize`, `pos_filter`, `min_freq`, and `ngrams` when creating your DTM.